## Neural Network Design and Training

In this notebook we will train a simple neural network on the MNIST handwritten digits dataset.

This notebook is the first step of the tutorial and focuses on **model training in TensorFlow/Keras**.  
In the next notebook (`quantization.ipynb`) we will use the trained model to perform model quantization using Vitis AI. 

The code used in this notebook relies on several helper modules located in the `src/` folder:

- `dataset.py` – dataset loading and preprocessing utilities  
- `models.py` – neural network architectures  
- `model_evaluation.py` – evaluation metrics and visualization tools

### Tutorial Goals

The objectives of this notebook are:

- Load and preprocess the MNIST dataset
- Split the dataset into training, validation, and test sets
- Train a simple neural network model
- Evaluate the trained model on unseen data

For simplicity, we will start with a **Multi-Layer Perceptron (MLP)**, which is one of the simplest neural network architectures.

After that, you can also play with a simple Convolutional Neural Network (CNN), which is more suitable for image data. 



### 1. Dataset loading and splitting

The MNIST dataset is a widely used benchmark in machine learning and computer vision. It contains 70,000 grayscale images of handwritten digits (0–9). Each image has a resolution of 28 × 28 pixels. 
The dataset is typically divided into 60,000 images for training and 10,000 images for testing. In this tutorial, we limit the test dataset to 5,000 images in order to speed up evaluation.



Before loading the dataset, we import the required libraries:

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import random

seed = 42
os.environ["PYTHONHASHSEED"] = str(seed)
random.seed(seed)
np.random.seed(seed)
tf.keras.utils.set_random_seed(seed)

The file `./src/dataset.py` contains helper functions used to load and preprocess the MNIST dataset. In particular, it provides two functions:

- `load_mnist()`  
  Loads the dataset and optionally a performs few preprocessing steps such as normalization, reshaping, and label encoding.

- `split_train_val()`  
  Splits the training set into **training** and **validation** subsets.

In [ ]:
from src.dataset import load_mnist, split_train_val

# Load data and select only 5,000 images for the test dataset
# This function also normalizes the dataset. Pixel values are normalized from [0,255] to [0,1]
(x_train, y_train), (x_test, y_test) = load_mnist(
    limit_test=5000,
    normalize=True,
    add_channel_dim=True,
    one_hot=True
)

# Train / validation split, with val_size = 10,000 samples
x_train, x_val, y_train, y_val = split_train_val(x_train, y_train, val_size=10000)
print(
    "Dataset shapes:\n"
    f"Train: {(x_train.shape, y_train.shape)}\n"
    f"Validation: {(x_val.shape, y_val.shape)}\n"
    f"Test: {(x_test.shape, y_test.shape)}"
)

In [ ]:
#In this section, you can plot some samples from the training and testing datasets

fig, axes = plt.subplots(2, 5, figsize=(12,5))

for i in range(5):
    axes[0,i].imshow(x_train[i].squeeze(-1), cmap="gray")
    axes[0,i].set_title(f"Train: {np.argmax(y_train[i])}")
    axes[0,i].axis("off")

    axes[1,i].imshow(x_test[i].squeeze(-1), cmap="gray")
    axes[1,i].set_title(f"Test: {np.argmax(y_test[i])}")
    axes[1,i].axis("off")

plt.tight_layout()
plt.show()

### 2. Model design and training

In this section we define and train our neural network model. 
The architectures used in this tutorial are implemented in the file `./scr/models.py`, which contains two models:
- `MPL` - a simple fully connected neural network
- `custom_CNN` - a small convolutional neural network

To make the notebook flexible, we use a model factory dictionary that allows us to easily switch between architectures.


In [ ]:
from src.models import MLP, custom_CNN

MODEL_FACTORY = {
    "MLP": MLP,
    "CNN": custom_CNN,
}

In [ ]:
#Choose your model and plot its structure

model_name = "MLP"   

N_classes = y_train.shape[1]

model = MODEL_FACTORY[model_name](
    input_shape=x_train.shape[1:],
    N_classes=N_classes
)

model.summary()

In [ ]:
#Choose the hyperparameters for the training
#Learning rate - controls how large the gradient updates are
#Batch size - number of samples processed at once during training
#Epochs - number of passes over the entire training dataset


learn_rate=0.001
batch_size = 256
epochs=5

# Create folder if it does not exist
os.makedirs("results", exist_ok=True)

model_path = f"./results/{model_name}.h5"

We now compile and train the model.

The compilation step defines:

- the optimizer (Adam)
- the loss function (categorical cross-entropy)
- the evaluation metric (accuracy)

During training we also use a ModelCheckpoint callback to automatically save the best model according to the validation loss. After training, we visualize the training history (training and validation loss and accuracy). 

In [ ]:
import time
import pandas as pd
from tensorflow.keras.callbacks import ModelCheckpoint

#-------------------------------------------------------------
#-------------------------------------------------------------
tf.keras.backend.clear_session()
model = MODEL_FACTORY[model_name](input_shape=x_train.shape[1:], N_classes=y_train.shape[1])

# ------------------------------------------------------------
# Compile
# ------------------------------------------------------------
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=learn_rate),
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=False),
    metrics=["accuracy"]
)

# ------------------------------------------------------------
# Checkpoint
# ------------------------------------------------------------
checkpoint = ModelCheckpoint(
    model_path,
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=False
)

# ------------------------------------------------------------
# Dataset pipeline
# ------------------------------------------------------------
train_batches = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(x_train.shape[0], seed=seed, reshuffle_each_iteration=False)
    .batch(batch_size)
    .prefetch(1)
)

val_batches = (
    tf.data.Dataset.from_tensor_slices((x_val, y_val))
    .batch(batch_size)
    .prefetch(1)
)

# ------------------------------------------------------------
# Training
# ------------------------------------------------------------
print("Fitting the model...")
t = time.time()

results = model.fit(
    train_batches,
    validation_data=val_batches,
    epochs=epochs,
    callbacks=[checkpoint]
)

elapsed_time = time.time() - t
print(f"Training completed in {elapsed_time/60:.3f} minutes ({elapsed_time:.3f} seconds)")

# ------------------------------------------------------------
# History
# ------------------------------------------------------------
hist_df = pd.DataFrame(results.history)

# Loss plot
plt.figure(figsize=(10, 5))
plt.plot(hist_df["loss"], label="Training Loss")
plt.plot(hist_df["val_loss"], label="Validation Loss")
plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

# Accuracy plot
plt.figure(figsize=(10, 5))
plt.plot(hist_df["accuracy"], label="Training Accuracy")
plt.plot(hist_df["val_accuracy"], label="Validation Accuracy")
plt.title("Training and Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

# ------------------------------------------------------------
# Final metrics
# ------------------------------------------------------------
last_epoch = len(hist_df) - 1

train_acc  = hist_df["accuracy"][last_epoch]
val_acc    = hist_df["val_accuracy"][last_epoch]
train_loss = hist_df["loss"][last_epoch]
val_loss   = hist_df["val_loss"][last_epoch]

print(f"Training Loss: {train_loss:.6f}")
print(f"Validation Loss: {val_loss:.6f}")
print(f"Training Accuracy: {train_acc:.6f}")
print(f"Validation Accuracy: {val_acc:.6f}")

### 3. Model evaluation on test dataset

Finally, we evaluate the trained model on the test dataset, which contains images that were not used during training.
Evaluating on unseen data allows us to estimate how well the model generalizes to new inputs.

The file `src/model_evaluation.py` contains the evaluation utilities used in this section. These include:
- Accuracy - The proportion of correctly classified samples over the total number of samples.
- Confusion matrix - A table that shows how often each digit is predicted as each possible class. The diagonal elements correspond to correct predictions. The off-diagonal elements correspond to misclassifications.
- (Optionally) Recall and F1-score

In [ ]:
from src.evaluation import compute_metrics, plot_confusion_matrix

# Predictions
Y_pred = model.predict(x_test, verbose=0)
y_pred_classes = np.argmax(Y_pred, axis=1)

# True labels
y_true = np.argmax(y_test, axis=1) if np.ndim(y_test) > 1 else y_test

# Metrics
metrics = compute_metrics(y_true, y_pred_classes, compute_f1=False, compute_recall=False)

print("\nEvaluation metrics on test dataset:")
for name, value in metrics.items():
    print(f"{name}: {value:.5f}")

#Confusion matrix
Classes=[str(i) for i in range(10)]
# Confusion matrix
plt.figure(figsize=(7,6))
plot_confusion_matrix(
    y_true,
    y_pred_classes,
    classes=Classes,
    title="Confusion Matrix - Test dataset",
    normalize=True
)

### 4. Questions 

**Q1.** Using the model factory dictionary, try to train the CNN model. What happens to the number of model parameters and the classification accuracy when switching from the MLP model to the CNN model? How can you explain this behavior?

**Q2.** Why is it useful to keep the model small and efficient before applying quantization and deployment on edge hardware?

**Q3.** Try implementing a lightweight CNN model (`CNN_lightweight`) in which the number of convolutional filters is reduced by half compared to the original CNN architecture. How would you modify the model definition to implement this change? After training the model for 5 epochs, how do the number of parameters and the test accuracy change compared to the original CNN?